# MOFI Tutorial 01: Kidney Organoid Differentiation (GSE213152)

This tutorial demonstrates MOFI's dual-modal dynamics reconstruction on kidney organoid differentiation data (RNA + ATAC).

The kidney organoid system features hierarchical developmental programs:
- Nephron Progenitor Cells (NPCs) → Developing Tubules (dev_TUB) / Developing Podocytes (dev_POD)
- dev_TUB → Proximal Tubule (PT), Loop of Henle (LOH), Distal Nephron (DN)
- dev_POD → Terminally differentiated Podocytes (POD)

## 1. Load Packages

In [ ]:
import sys
from pathlib import Path

src_path = Path("../src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import torch

import CytoBridge.pp as cb_pp
import CytoBridge.tl as cb_tl
from CytoBridge.utils.utils import set_seed, load_model_from_adata
from CytoBridge.Map.tl.trainer import main as map_train_main
from CytoBridge.Map.tl.transport_factory import build_transport_map

set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

## 2. Load Data

GSE213152 contains paired RNA and ATAC data from kidney organoid differentiation across 5 time points.

In [ ]:
adata_rna = sc.read_h5ad('../datasets/GSE213152/GSE213152_rna_pca50.h5ad')
adata_atac = sc.read_h5ad('../datasets/GSE213152/GSE213152_atac_gene_pca50.h5ad')

print(f"RNA data: {adata_rna.shape}")
print(f"ATAC data: {adata_atac.shape}")
print(f"\nRNA time points: {sorted(adata_rna.obs['time_point_processed'].unique())}")
print(f"ATAC time points: {sorted(adata_atac.obs['time_point_processed'].unique())}")
print(f"\nRNA cell types: {adata_rna.obs['cell_type'].unique().tolist()}")

## 3. Train Cross-Omics Mapper (SQCM)

The Symmetric Quad-path Cross-omics Mapper (SQCM) learns bidirectional mappings between RNA and ATAC latent spaces.

In [ ]:
import argparse

MAP_OUTPUT = '../Map_output/GSE213152'

map_args = argparse.Namespace(
    rna_adata_path='../datasets/GSE213152/GSE213152_rna_pca50.h5ad',
    protein_adata_path='../datasets/GSE213152/GSE213152_atac_gene_pca50.h5ad',
    latent_key='X_latent',
    save_path=MAP_OUTPUT,
    hidden_dim=32,
    num_layers=4,
    lr=0.001,
    batch_size=512,
    epochs=1000,
    earlystop=100,
    lossweight11=1.0,
    lossweight22=1.0,
    lossweight12=20.0,
    lossweight21=20.0,
    lossweight_z=100.0,
    auto_switch_cross=True,
    use_cross_recon=False,
    patience_stage1=20,
    patience_stage2=30,
    num_workers=0,
    device=device,
    seed=42,
    loss_plot_ext='png',
)

# Train the mapper
map_train_main(map_args)
print(f"\nMap model saved to: {MAP_OUTPUT}/best_model.pt")

## 4. Train Dual-Modal Dynamics

Now we train MOFI's dynamics model with cross-omics constraints. The `fit()` function with `adata_sec` enables synchronized unbalanced optimal transport.

In [ ]:
# Update config with actual map model path
config_path = '../examples/configs/GSE213152/unbalanced_ot_gse213152_12_cycle.yaml'

# Train dual-modal dynamics
adata = cb_tl.fit(
    adata_rna,
    config=config_path,
    adata_sec=adata_atac,
    device=device
)

## 5. Analyze Results

In [ ]:
# Inspect model outputs
print("Model components:", list(adata.uns['all_model']['model_config']['components']))
print(f"Velocity shape: {adata.obsm['velocity_latent'].shape}")
print(f"Growth rate shape: {adata.obsm['growth_rate'].shape}")

# Cell type composition over time
ct_time = adata.obs.groupby(['time_point_processed', 'cell_type']).size().unstack(fill_value=0)
ct_time_norm = ct_time.div(ct_time.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(8, 4))
ct_time_norm.plot(kind='area', stacked=True, ax=ax, alpha=0.8)
ax.set_xlabel('Time')
ax.set_ylabel('Cell type proportion')
ax.set_title('Cell type composition dynamics (RNA space)')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig('../tutorial/save_results/GSE213152_celltype_dynamics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Growth rate visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

scatter1 = axes[0].scatter(
    adata.obsm['X_latent'][:, 0], adata.obsm['X_latent'][:, 1],
    c=adata.obs['time_point_processed'], cmap='viridis', s=3, alpha=0.5
)
plt.colorbar(scatter1, ax=axes[0], label='Time')
axes[0].set_title('RNA latent space (colored by time)')

scatter2 = axes[1].scatter(
    adata.obsm['X_latent'][:, 0], adata.obsm['X_latent'][:, 1],
    c=adata.obsm['growth_rate'].flatten(), cmap='RdBu_r', s=3, alpha=0.5
)
plt.colorbar(scatter2, ax=axes[1], label='Growth rate')
axes[1].set_title('Inferred growth rate')

plt.tight_layout()
plt.savefig('../tutorial/save_results/GSE213152_growth_rate.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Full Pipeline (Optional)

For the complete pipeline including Map inference, direction selection, evaluation, and joint visualization, use the command-line pipeline:

```bash
cd /path/to/MOFI
python examples/scripts/run_pipeline.py --config examples/configs/GSE213152/pipeline_GSE213152_12.yaml
```

This will run:
1. Map model training
2. Map inference (4 modes: 11, 12, 21, 22)
3. Direction auto-selection
4. Dual-modal dynamics training
5. Single-space and cross-space evaluation (W1, TMV)
6. Trajectory visualization with MOFA/GAUDI joint plots

## Summary

This tutorial demonstrated MOFI's dual-modal workflow for kidney organoid differentiation:
1. **Load** paired RNA + ATAC data
2. **Train** cross-omics mapper (SQCM)
3. **Train** synchronized dynamics with cross-omics constraints
4. **Analyze** velocity fields, growth rates, and cell type dynamics

The synchronized unbalanced optimal transport ensures that inferred trajectories remain biologically consistent across both RNA and ATAC spaces.